# NVIDIA Nemotron Reasoning Challenge — Kaggle submission builder

This notebook runs the full pipeline (EDA → data prep → LoRA training → package) and produces **submission.zip**.

## Input dataset you must add

1. **Competition data** (required)  
   - Go to the [NVIDIA Nemotron Model Reasoning Challenge](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge) page.
   - Click **"Add Data"** and add the **competition dataset** (often named like `nvidia-nemotron-model-reasoning-challenge`).  
   - It must contain **train.csv** (columns: `id`, `prompt`, `answer`) and **test.csv** (columns: `id`, `prompt`).

2. **Project code** (required)  
   - **Option A (no Internet)**: Add a Kaggle Dataset that contains the project (e.g. `scripts/`, `run_all.py`, `requirements.txt`). In the next cell set `SOURCE_CODE_INPUT_PATH` to its path (e.g. `/kaggle/input/datasets/your-user/your-dataset-name`).  
   - **Option B**: Set `GITHUB_REPO` to a public repo URL and enable Internet; the notebook will clone it if `SOURCE_CODE_INPUT_PATH` is empty.

3. **Nemotron model** (for offline runs): If you run with Internet off, add a Kaggle Dataset containing the **NVIDIA-Nemotron-3-Nano-30B-A3B-BF16** model (same layout as on Hugging Face). Set `NEMOTRON_INPUT_PATH` in the config to that path. With Internet on, the model is downloaded automatically.

## Settings

- **GPU**: Enable GPU (Settings → Accelerator → GPU) so LoRA training runs.
- **Internet / GTX:** Some accelerators (e.g. **GTX** on this competition) **cannot enable Internet** — Kaggle blocks it. Then you **must** use: (1) a **Kaggle Dataset of pip wheels** → set `PIP_FIND_LINKS_DIR` and `PIP_OFFLINE=True`; (2) **`NEMOTRON_INPUT_PATH`** to a local copy of the Nemotron model (HF layout). Build wheels on a Linux+CUDA machine with `pip download -r requirements-kaggle-peft.txt -d wheels` (same Python major as Kaggle, e.g. 3.12). On accelerators that **do** allow Internet, leave `PIP_OFFLINE=False` and you can use PyPI.
- **Secrets** (optional): To use real training data + CoT generation, add `ANTHROPIC_API_KEY` in Notebook Secrets and set `USE_SYNTHETIC_ONLY = False`.

## Pip vs GPU (important)

**`pip install` never uses the GPU** — only **CPU** and **disk**. **0% GPU during install is normal.** T4s are used when **training** starts (Phase 3).

## Disk (~57 GiB on `/kaggle/working`)

If usage **exceeds the max** (sidebar), things get slow or fail. Large uses: **HF model cache**, **pip cache**, **`hf_offload/`**. Prefer **`NEMOTRON_INPUT_PATH`** (model on input dataset) to avoid a huge duplicate cache; you can remove `/kaggle/working/.pip-cache` or `hf_offload` to free space. The next code cell sets **`HF_HOME` under `/kaggle/working/.hf`** so the Hub cache location is clear.

In [ ]:
# ========== CONFIG (edit these) ==========
# Leave empty to auto-detect from /kaggle/input (first folder with train.csv + test.csv).
COMPETITION_INPUT_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge"

# Project code: set this to your Kaggle Dataset path (no Internet needed). Leave empty to clone from GitHub.
SOURCE_CODE_INPUT_PATH = "/kaggle/input/datasets/sebmontreal/nvidia-nemotron-reasoning-challenge-source"

# GitHub repo URL (only used if SOURCE_CODE_INPUT_PATH is empty). Must be public; enable Internet.
GITHUB_REPO = "https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge"

# Use only synthetic data (no API calls). Set False if you added ANTHROPIC_API_KEY in Secrets.
USE_SYNTHETIC_ONLY = True

# Skip evaluation step to save time (submission.zip does not need it).
SKIP_EVAL = True

# Use PEFT only for training (skip Unsloth). Set True on Kaggle to avoid "no kernel image" CUDA errors on newer GPUs (e.g. Blackwell).
USE_PEFT_ONLY = True

# Nemotron model path: set to your Kaggle Dataset path (avoids HF download + saves disk). Run !ls /kaggle/input to see actual paths.
NEMOTRON_INPUT_PATH = "/kaggle/input/nemotron-model"

# Pip: GTX on this competition = Internet disabled by Kaggle — set PIP_OFFLINE=True + PIP_FIND_LINKS_DIR (wheel dataset).
PIP_REQUIREMENTS_FILE = "requirements-kaggle-peft.txt"  # Smaller set for --peft-only + synthetic-only (no Unsloth). Full list: "requirements.txt"
SKIP_PIP_INSTALL = False  # True = skip pip (only if base image already has everything)
PIP_FIND_LINKS_DIR = ""  # Required for GTX offline: path to folder of .whl files (upload as a Kaggle Dataset)
PIP_OFFLINE = False  # Set True for GTX/no-Internet; must match PIP_FIND_LINKS_DIR with wheels for PIP_REQUIREMENTS_FILE

# Faster repeat installs (same Kaggle session): pip cache in working dir + skip mamba reinstall once deps work
PIP_USE_WORKING_CACHE = True  # PIP_CACHE_DIR=/kaggle/working/.pip-cache (helps if you re-run the install cell without restart)
SKIP_MAMBA_REINSTALL = False  # Set True after first successful run to skip slow mamba-ssm/causal-conv1d --force-reinstall (~minutes)
SKIP_TORCHVISION_SYNC = False  # Set True only if torchvision already matches torch (rare); sync fixes torchvision::nms / PreTrainedModel import errors

In [ ]:
# Suppress traitlets/papermill FutureWarning (Kaggle environment; safe to ignore)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="traitlets")

## 1. Find competition data and prepare working dir

In [ ]:
import os
import shutil

WORKING = "/kaggle/working"
INPUT = "/kaggle/input"
DATA_DIR = os.path.join(WORKING, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Hugging Face cache on working volume (predictable; pip/install is CPU-only — GPUs stay idle until training)
_hf = os.path.join(WORKING, ".hf")
os.makedirs(_hf, exist_ok=True)
os.environ.setdefault("HF_HOME", _hf)
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", os.path.join(_hf, "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", os.path.join(_hf, "hub"))

# Resolve competition data path (train.csv + test.csv at root, or in data/ subfolder)
def _find_comp_path(base):
    if os.path.isfile(os.path.join(base, "train.csv")) and os.path.isfile(os.path.join(base, "test.csv")):
        return base
    data_sub = os.path.join(base, "data")
    if os.path.isdir(data_sub) and os.path.isfile(os.path.join(data_sub, "train.csv")) and os.path.isfile(os.path.join(data_sub, "test.csv")):
        return data_sub
    return None

comp_path = None
if COMPETITION_INPUT_PATH and os.path.isdir(COMPETITION_INPUT_PATH):
    comp_path = _find_comp_path(COMPETITION_INPUT_PATH)
if comp_path is None:
    for name in os.listdir(INPUT):
        path = os.path.join(INPUT, name)
        if os.path.isdir(path):
            comp_path = _find_comp_path(path)
            if comp_path:
                break
if comp_path is None:
    raise FileNotFoundError(
        "Competition data not found. Add the competition dataset (with train.csv and test.csv) or the source dataset (has data/train.csv, data/test.csv) to this notebook."
    )
print(f"Using competition data: {comp_path}")

for f in ["train.csv", "test.csv"]:
    src = os.path.join(comp_path, f)
    dst = os.path.join(DATA_DIR, f)
    shutil.copy2(src, dst)
    print(f"Copied {f}")

print(f"Data dir: {DATA_DIR}")
print(os.listdir(DATA_DIR))

## 2. Get project code (clone or copy from input)

In [ ]:
import subprocess

code_ready = False

# Prefer code from Kaggle Dataset (no Internet needed)
if SOURCE_CODE_INPUT_PATH and os.path.isdir(SOURCE_CODE_INPUT_PATH):
    scripts_marker = os.path.join(SOURCE_CODE_INPUT_PATH, "scripts", "01_eda.py")
    if os.path.isfile(scripts_marker):
        print(f"Copying code from dataset: {SOURCE_CODE_INPUT_PATH}")
        for item in os.listdir(SOURCE_CODE_INPUT_PATH):
            if item == "data":
                continue
            src = os.path.join(SOURCE_CODE_INPUT_PATH, item)
            dst = os.path.join(WORKING, item)
            if os.path.exists(dst):
                if os.path.isdir(dst):
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            if os.path.isdir(src):
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
        code_ready = True
        print("Code copied.")
    else:
        print(f"SOURCE_CODE_INPUT_PATH set but scripts/01_eda.py not found at {SOURCE_CODE_INPUT_PATH}")

if not code_ready and GITHUB_REPO and GITHUB_REPO.strip():
    # Clone from GitHub (enable Internet in notebook settings)
    repo = GITHUB_REPO.strip().rstrip("/")
    if not repo.endswith(".git"):
        repo = repo + ".git"
    print(f"Cloning {repo} into {WORKING}...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", repo, "repo_temp"],
        cwd=WORKING,
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        # Move contents to working (avoid nested folder); keep our data/
        for item in os.listdir(os.path.join(WORKING, "repo_temp")):
            if item == "data":
                continue
            src = os.path.join(WORKING, "repo_temp", item)
            dst = os.path.join(WORKING, item)
            if os.path.exists(dst):
                if os.path.isdir(dst):
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            shutil.move(src, dst)
        shutil.rmtree(os.path.join(WORKING, "repo_temp"), ignore_errors=True)
        code_ready = True
        print("Clone done.")
    else:
        print("Clone failed:", result.stderr or result.stdout)

if not code_ready:
    # Look for code in an input dataset (zip or folder with scripts/)
    for name in os.listdir(INPUT):
        path = os.path.join(INPUT, name)
        if not os.path.isdir(path):
            continue
        scripts_path = os.path.join(path, "scripts", "01_eda.py")
        if os.path.isfile(scripts_path):
            print(f"Copying code from input dataset: {name}")
            for item in os.listdir(path):
                src = os.path.join(path, item)
                dst = os.path.join(WORKING, item)
                if item == "data":
                    continue
                if os.path.exists(dst):
                    if os.path.isdir(dst):
                        shutil.rmtree(dst)
                    else:
                        os.remove(dst)
                shutil.copytree(src, dst) if os.path.isdir(src) else shutil.copy2(src, dst)
            code_ready = True
            break
        # Try zip
        for f in os.listdir(path):
            if f.endswith(".zip"):
                zip_path = os.path.join(path, f)
                print(f"Unzipping {zip_path} to {WORKING}")
                shutil.unpack_archive(zip_path, WORKING)
                code_ready = os.path.isfile(os.path.join(WORKING, "scripts", "01_eda.py"))
                if code_ready:
                    break
        if code_ready:
            break

if not code_ready:
    raise FileNotFoundError(
        "Project code not found. Set GITHUB_REPO to your public repo URL, or add a Kaggle Dataset that contains the project (scripts/, run_all.py, requirements.txt)."
    )

# Ensure competition data is in place (in case clone/zip had a data/ folder)
for f in ["train.csv", "test.csv"]:
    src = os.path.join(comp_path, f)
    dst = os.path.join(DATA_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("Project files:", os.listdir(WORKING))

## 3. Install dependencies

**GTX / no Internet:** set `PIP_OFFLINE=True` and `PIP_FIND_LINKS_DIR` to your wheel dataset; the PyPI check is skipped. **With Internet** (other GPUs): leave `PIP_OFFLINE=False`; the cell checks PyPI then runs `pip install -r` (see `PIP_REQUIREMENTS_FILE` in the config cell).

**Speed on T4:** First install is slow (many wheels + optional mamba rebuild). Use `PIP_USE_WORKING_CACHE=True` and re-run only the install cell in the **same session** to hit the cache. After mamba imports work once, set **`SKIP_MAMBA_REINSTALL=True`** to skip the extra reinstall. For repeated runs across commits, upload a **wheel dataset** and use `PIP_FIND_LINKS_DIR` + `PIP_OFFLINE=True` (or run training in a persistent VM).

In [ ]:
import os
import subprocess
import sys
import urllib.request

os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")
if PIP_USE_WORKING_CACHE:
    pip_cache = os.path.join(WORKING, ".pip-cache")
    os.makedirs(pip_cache, exist_ok=True)
    os.environ["PIP_CACHE_DIR"] = pip_cache

if SKIP_PIP_INSTALL:
    print("SKIP_PIP_INSTALL=True — skipping pip. If import fails later, install deps or set PIP_FIND_LINKS_DIR.")
else:
    if not PIP_OFFLINE:
        try:
            urllib.request.urlopen("https://pypi.org/simple/trl/", timeout=15)
            print("Network OK: can reach PyPI.")
        except Exception as e:
            raise RuntimeError(
                "This session cannot reach PyPI (DNS failed).\n\n"
                "If you are on GTX for this competition, Kaggle often **blocks Internet entirely** — you cannot fix this with Settings. "
                "Use PIP_OFFLINE=True + a wheel dataset (PIP_FIND_LINKS_DIR); see README 'GTX' section.\n\n"
                "Otherwise (GPU that allows Internet): save the notebook with Internet ON, then Save Version / Run All.\n\n"
                "Workarounds: SKIP_PIP_INSTALL=True if deps exist, or wheel dataset + PIP_FIND_LINKS_DIR + PIP_OFFLINE=True.\n\n"
                f"Underlying error: {e}"
            ) from e

    req_path = (
        PIP_REQUIREMENTS_FILE
        if os.path.isabs(PIP_REQUIREMENTS_FILE)
        else os.path.join(WORKING, PIP_REQUIREMENTS_FILE)
    )
    if not os.path.isfile(req_path):
        raise FileNotFoundError(
            f"Missing {req_path}. Add requirements-kaggle-peft.txt to your code dataset or set PIP_REQUIREMENTS_FILE."
        )
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--prefer-binary",
        "-r",
        req_path,
    ]
    if PIP_FIND_LINKS_DIR and os.path.isdir(PIP_FIND_LINKS_DIR):
        cmd += ["--find-links", PIP_FIND_LINKS_DIR]
    if PIP_OFFLINE:
        if not (PIP_FIND_LINKS_DIR and os.path.isdir(PIP_FIND_LINKS_DIR)):
            raise RuntimeError("PIP_OFFLINE=True requires PIP_FIND_LINKS_DIR to an existing directory of .whl files.")
        cmd += ["--no-index"]

    rc = subprocess.run(cmd).returncode
    if rc != 0:
        raise RuntimeError(
            "pip install failed. See pip output above. With network: check versions/build errors. "
            "Without network: use a complete wheelhouse and PIP_OFFLINE=True."
        )
    print("Requirements installed.")
    # torchvision must match the *installed* torch or transformers fails (operator torchvision::nms does not exist).
    if not SKIP_TORCHVISION_SYNC:
        print("Syncing torchvision to match installed torch (PyTorch wheel index)...")
        try:
            import torch as _torch_for_cu
            _cv = getattr(_torch_for_cu.version, "cuda", None)
            if _cv:
                _p = str(_cv).split(".")
                _cu = f"cu{_p[0]}{_p[1]}" if len(_p) >= 2 else None
                _idx = f"https://download.pytorch.org/whl/{_cu}" if _cu else None
            else:
                _idx = None
        except Exception:
            _idx = None
        _tv_cmd = [
            sys.executable, "-m", "pip", "install", "-q", "--prefer-binary",
            "torchvision", "--force-reinstall",
        ]
        if _idx:
            _tv_cmd += ["--extra-index-url", _idx]
        _tvr = subprocess.run(_tv_cmd, cwd=WORKING).returncode
        if _tvr != 0 and _idx:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "torchvision", "--force-reinstall"],
                cwd=WORKING,
            )
    else:
        print("SKIP_TORCHVISION_SYNC=True — skipped torchvision reinstall.")
    # mamba_ssm .so must match torch; skip on repeat runs (SKIP_MAMBA_REINSTALL=True) to save several minutes.
    if not SKIP_MAMBA_REINSTALL:
        print("Reinstalling mamba-ssm + causal-conv1d for torch ABI (set SKIP_MAMBA_REINSTALL=True next time if imports already work)...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--prefer-binary",
                "mamba-ssm",
                "causal-conv1d",
                "--force-reinstall",
            ],
            cwd=WORKING,
        )
    else:
        print("SKIP_MAMBA_REINSTALL=True — skipped mamba-ssm/causal-conv1d reinstall.")

### Model path check
Run this cell to see your Kaggle input paths and the Nemotron model location.

In [ ]:
# Run this to see your input paths — use the path containing config.json for NEMOTRON_INPUT_PATH
print("Contents of /kaggle/input:")
for name in sorted(os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []):
    p = os.path.join("/kaggle/input", name)
    print(f"  {p}")
    if os.path.isdir(p):
        for sub in sorted(os.listdir(p))[:8]:
            sp = os.path.join(p, sub)
            ok = " [model ✓]" if (os.path.isdir(sp) and os.path.isfile(os.path.join(sp, "config.json"))) else ""
            print(f"    {sub}{ok}")
print("\nNEMOTRON_INPUT_PATH =", repr(NEMOTRON_INPUT_PATH))

## 4. Run pipeline

In [ ]:
import sys

def _find_model_dir(base):
    """Return path to dir with config.json, or None."""
    if not base or not os.path.isdir(base):
        return None
    if os.path.isfile(os.path.join(base, "config.json")):
        return base
    for name in os.listdir(base):
        sub = os.path.join(base, name)
        if os.path.isdir(sub) and os.path.isfile(os.path.join(sub, "config.json")):
            return sub
    return None

def _scan_for_model():
    """Recursively scan /kaggle/input for a dir containing config.json (Kaggle paths vary)."""
    found = []
    for root, dirs, _ in os.walk("/kaggle/input", topdown=True):
        for d in dirs:
            p = os.path.join(root, d)
            if os.path.isfile(os.path.join(p, "config.json")):
                found.append(p)
        if found:
            break
    return found[0] if found else None

# Resolve Nemotron model path (Kaggle dataset paths vary, e.g. /kaggle/input/datasets/owner/dataset-name)
_nemotron_path = None
_candidates = [
    NEMOTRON_INPUT_PATH,
    "/kaggle/input/nemotron-model",
    "/kaggle/input/datasets/sebmontreal/nemotron-model",
]
for c in _candidates:
    if c and os.path.isdir(c):
        _nemotron_path = _find_model_dir(c)
        if _nemotron_path:
            break

if not _nemotron_path:
    _nemotron_path = _scan_for_model()

if not _nemotron_path:
    try:
        import kagglehub
        _kh_path = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
        if _kh_path and os.path.isfile(os.path.join(_kh_path, "config.json")):
            _nemotron_path = _kh_path
    except Exception as e:
        print(f"kagglehub: {e}")

if not _nemotron_path or not os.path.isfile(os.path.join(_nemotron_path, "config.json")):
    print("Contents of /kaggle/input:")
    for name in (os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []):
        p = os.path.join("/kaggle/input", name)
        print(f"  {p}/")
        if os.path.isdir(p):
            for sub in (os.listdir(p)[:5] if os.path.isdir(p) else []):
                print(f"    {sub}")
    raise RuntimeError(
        "Nemotron model not found (no config.json). Set NEMOTRON_INPUT_PATH to the path shown above."
    )

os.environ["NEMOTRON_MODEL_PATH"] = _nemotron_path
os.environ["HF_HUB_OFFLINE"] = "1"
print(f"Using Nemotron model: {_nemotron_path}")

args = ["python", "run_all.py"]
if USE_SYNTHETIC_ONLY:
    args.append("--synthetic-only")
if SKIP_EVAL:
    args.append("--skip-eval")
if USE_PEFT_ONLY:
    args.append("--peft-only")

print("Running:", " ".join(args))
result = subprocess.run(args, cwd=WORKING, env=os.environ.copy())
if result.returncode != 0:
    sys.exit(result.returncode)
print("Pipeline finished.")

## 5. Verify submission

In [ ]:
sub_path = os.path.join(WORKING, "submission.zip")
if os.path.isfile(sub_path):
    print(f"submission.zip size: {os.path.getsize(sub_path):,} bytes")
    print("You can submit this file to the competition.")
else:
    print("submission.zip not found. Check pipeline logs above.")